In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
path = f"/Volumes/adb_caaresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check if the path or parent directory exists
try:
    dbutils.fs.ls(path)
except Exception:
    print(f"No new files received. Path does not exist: {path}")
    dbutils.notebook.exit("No new files received")

if source_file_format == "csv":
    landing_df = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
else:
    landing_df = spark.read.parquet(path)
#display(landing_df)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = (landing_df
    .withColumn("landing_timestamp", lit(landing_timestamp))
    .withColumn("insert_timestamp", current_timestamp())
)

#display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caaresync.bronze.{table_name}")